# Monte Carlo Methods

## Introduction

Monte Carlo (MC) methods learn value functions and optimal policies directly from **sampled experience**. Unlike dynamic programming, they do not require the transition model $p(s',r\mid s,a)$ or a probability distribution over all possible outcomes.

An episode is a complete trajectory ending at a terminal state:

$$
S_0,A_0,R_1,S_1,A_1,R_2,\ldots,S_{T-1},A_{T-1},R_T,S_T.
$$

The return observed after visiting time step $t$ is

$$
G_t = \sum_{k=0}^{T-t-1}\gamma^k R_{t+k+1}.
$$

MC methods estimate values by averaging these observed returns across episodes:

$$
v_\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s],
\qquad
q_\pi(s,a)=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a].
$$

Because $G_t$ is known only after termination, classical MC methods update values **after complete episodes**. They therefore apply naturally to episodic tasks and do not bootstrap from current value estimates.

- **Prediction:** estimate $v_\pi$ or $q_\pi$ from episodes generated under a fixed policy $\pi$.
- **Control:** alternate value estimation and policy improvement through generalized policy iteration (GPI).

For control, action values are central: without a model, the agent cannot evaluate an action by enumerating its possible successor states, but it can estimate its quality from sampled returns.

## Monte Carlo Prediction

Given a fixed policy $\pi$, prediction means estimating its state-value function

$$
v_\pi(s)=\mathbb{E}_\pi[G_t\mid S_t=s].
$$

Every time an episode visits state $s$, the return following that visit is one sample of the random variable $G_t\mid S_t=s$. If the collected returns are $G^{(1)}(s),\ldots,G^{(N(s))}(s)$, then the Monte Carlo estimate is their sample mean:

$$
V(s)=\frac{1}{N(s)}\sum_{i=1}^{N(s)}G^{(i)}(s).
$$

As $N(s)\to\infty$, the law of large numbers gives

$$
V(s)\longrightarrow \mathbb{E}_\pi[G_t\mid S_t=s]=v_\pi(s),
$$

provided episodes are generated under $\pi$ and state $s$ continues to be visited.

### First-Visit and Every-Visit MC

A state may occur multiple times in one episode.

- **First-visit MC:** uses only the return following the first occurrence of $s$ in each episode.
- **Every-visit MC:** uses the return following every occurrence of $s$.

Both converge to $v_\pi(s)$ under standard assumptions. First-visit MC produces at most one sample per state per episode; every-visit MC reuses all visits.

### Incremental Mean

The complete list of returns need not be stored. After observing the $n$th return $G_n(s)$, update

$$
V_n(s)
=V_{n-1}(s)+\frac{1}{n}\left[G_n(s)-V_{n-1}(s)\right].
$$

The difference $G_n(s)-V_{n-1}(s)$ is the estimation error of the new sample, and $1/n$ maintains the exact sample average.

### First-Visit MC Algorithm

```text
Input: policy pi
Initialize V(s) arbitrarily and N(s) = 0 for every state s

Repeat for each episode:
    Generate S_0, A_0, R_1, ..., S_T by following pi
    G = 0

    For t = T - 1, T - 2, ..., 0:
        G = R_(t+1) + gamma * G

        If S_t does not occur in S_0, ..., S_(t-1):
            N(S_t) = N(S_t) + 1
            V(S_t) = V(S_t) + [G - V(S_t)] / N(S_t)
```

The backward recursion computes the return efficiently because

$$
G_t=R_{t+1}+\gamma G_{t+1}.
$$

The first-visit condition is chronological: an update is made only if $S_t$ did not appear at an earlier time in the same episode.

### Monte Carlo versus Dynamic Programming

A DP backup computes an exact one-step expectation using the environment model:

$$
V(s)\leftarrow
\sum_a\pi(a\mid s)
\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V(s')\right].
$$

An MC backup instead uses one complete sampled trajectory and moves $V(S_t)$ toward its observed return:

$$
V(S_t)\leftarrow V(S_t)+\alpha\left[G_t-V(S_t)\right].
$$

Therefore:

- **DP averages over all one-step outcomes; MC samples one outcome sequence to termination.**
- **DP bootstraps:** its target contains the estimate $V(S_{t+1})$.
- **MC does not bootstrap:** $G_t$ is computed entirely from observed rewards.
- MC requires no transition probabilities, even when the exact expectation is expensive or impractical to compute.

The estimate of one state can be updated without computing values for all other states. Its cost depends on sampled episodes containing that state, not on $|\mathcal{S}|$. This makes MC useful when only a small reachable subset of a large state space matters.

## Monte Carlo Estimation of Action Values

For a fixed policy $\pi$, the action-value function is

$$
q_\pi(s,a)
=\mathbb{E}_\pi[G_t\mid S_t=s,A_t=a].
$$

It is the expected return obtained by taking action $a$ in state $s$ and following $\pi$ afterward.

### Why State Values Are Insufficient Without a Model

If $v_\pi$ is known, improving the policy requires evaluating each candidate action:

$$
q_\pi(s,a)
=\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma v_\pi(s')\right].
$$

This calculation requires the transition model $p(s',r\mid s,a)$. Without that model, $v_\pi(s)$ says how good it is to be in $s$ under $\pi$, but it does not determine the consequence of choosing a different action $a$.

Estimating $q_\pi$ directly removes this dependency. Policy improvement then becomes

$$
\pi'(s)\in\arg\max_{a\in\mathcal{A}(s)}q_\pi(s,a),
$$

which requires only learned action values, not transition probabilities.

### Monte Carlo Estimator

A visit to $(s,a)$ occurs at time $t$ when $S_t=s$ and $A_t=a$. The return following that visit,

$$
G_t=R_{t+1}+\gamma R_{t+2}+\cdots+\gamma^{T-t-1}R_T,
$$

is one sample of $G_t\mid S_t=s,A_t=a$. After observing $N(s,a)$ such returns,

$$
Q(s,a)
=\frac{1}{N(s,a)}
\sum_{i=1}^{N(s,a)}G^{(i)}(s,a).
$$

Equivalently, the sample mean can be updated incrementally:

$$
N(s,a)\leftarrow N(s,a)+1,
$$

$$
Q(s,a)\leftarrow Q(s,a)
+\frac{1}{N(s,a)}\left[G_t-Q(s,a)\right].
$$

- **First-visit MC:** uses only the return after the first occurrence of $(s,a)$ in an episode.
- **Every-visit MC:** uses the return after every occurrence of $(s,a)$.

If every state-action pair is visited infinitely often, then

$$
N(s,a)\to\infty
\quad\Longrightarrow\quad
Q(s,a)\to q_\pi(s,a).
$$

### Exploration Is Necessary

An unvisited action has no return samples, so its value cannot be estimated. A policy that always selects its current greedy action may permanently exclude alternatives that are actually better. Reliable control therefore requires sufficient exploration:

$$
N(s,a)\to\infty
\qquad
\text{for every relevant }(s,a).
$$

**Exploring starts** enforces coverage by assigning nonzero probability to every possible initial pair:

$$
\Pr(S_0=s,A_0=a)>0
\qquad \forall(s,a).
$$

This assumption is mathematically convenient but often unrealistic because the agent may not control the initial state. A practical alternative is a stochastic policy satisfying

$$
\pi(a\mid s)>0
\qquad \forall a\in\mathcal{A}(s),
$$

so every action retains a nonzero chance of being sampled whenever state $s$ is visited.

## Monte Carlo Control

Monte Carlo estimation can be used in control to approximate optimal policies. According to generalized policy iteration (GPI), an approximate policy and an approximate value function are maintained. The value function is repeatedly altered to more closely approximate the value function for the current policy, and the policy is repeatedly improved with respect to the current value function.

A Monte Carlo version of classical policy iteration performs alternating complete steps of policy evaluation and policy improvement:

$$
\pi_0
\xrightarrow{E} q_{\pi_0}
\xrightarrow{I} \pi_1
\xrightarrow{E} q_{\pi_1}
\xrightarrow{I} \pi_2
\xrightarrow{E}\cdots
\xrightarrow{I}\pi_*
\xrightarrow{E}q_*.
$$

Here, $\xrightarrow{E}$ denotes complete policy evaluation and $\xrightarrow{I}$ denotes complete policy improvement.

For the moment, assume that episodes are generated with exploring starts and that policy evaluation is performed with an infinite number of episodes. Under these assumptions, the Monte Carlo methods compute each $q_{\pi_k}$ exactly.

### Policy Improvement

For any action-value function $q$, the corresponding greedy policy chooses an action with maximal action value:

$$
\pi(s)\doteq\arg\max_a q(s,a).
$$

Each policy $\pi_{k+1}$ is greedy with respect to $q_{\pi_k}$. For all $s\in\mathcal{S}$,

$$
\begin{aligned}
q_{\pi_k}\bigl(s,\pi_{k+1}(s)\bigr)
&=q_{\pi_k}\left(s,\arg\max_a q_{\pi_k}(s,a)\right)\\
&=\max_a q_{\pi_k}(s,a)\\
&\ge q_{\pi_k}\bigl(s,\pi_k(s)\bigr)\\
&=v_{\pi_k}(s).
\end{aligned}
$$

The policy improvement theorem then assures that each $\pi_{k+1}$ is uniformly better than, or just as good as, $\pi_k$. This process converges to the optimal policy and optimal value function in a finite number of steps.

### Episode-by-Episode Evaluation and Improvement

The assumption that policy evaluation operates on an infinite number of episodes cannot be obtained in practice. Instead of completing policy evaluation before policy improvement, evaluation and improvement are alternated on an episode-by-episode basis. After each episode, the observed returns are used for policy evaluation, and the policy is improved at all states visited in the episode.

### Monte Carlo ES (Exploring Starts)

```text
Initialize:
    pi(s) arbitrarily, for all s in S
    Q(s, a) arbitrarily, for all s in S and a in A(s)
    Returns(s, a) as an empty list, for all s in S and a in A(s)

Loop forever, for each episode:
    Choose S_0 and A_0 randomly such that all pairs have probability > 0
    Generate an episode from S_0, A_0, following pi:
        S_0, A_0, R_1, ..., S_(T-1), A_(T-1), R_T
    G = 0

    For t = T - 1, T - 2, ..., 0:
        G = gamma * G + R_(t+1)

        Unless the pair S_t, A_t appears in
        S_0, A_0, S_1, A_1, ..., S_(t-1), A_(t-1):
            Append G to Returns(S_t, A_t)
            Q(S_t, A_t) = average(Returns(S_t, A_t))
            pi(S_t) = argmax_a Q(S_t, a)
```

In Monte Carlo ES, all returns for each state-action pair are accumulated and averaged. It is easy to see that Monte Carlo ES cannot converge to a suboptimal policy: if it did, the value function would eventually converge to the value function for that policy, causing the policy to change. Stability is achieved only when both the policy and the value function are optimal. However, convergence to this optimal fixed point has not been formally proved.

## Monte Carlo Control without Exploring Starts

To avoid the assumption of exploring starts, all actions must be selected infinitely often. There are two approaches:

- **On-policy methods** evaluate or improve the policy used to make decisions.
- **Off-policy methods** evaluate or improve a policy different from that used to generate the data.

### $\varepsilon$-Soft and $\varepsilon$-Greedy Policies

In on-policy control, the policy is generally soft:

$$
\pi(a\mid s)>0
\qquad
\text{for all }s\in\mathcal{S},\ a\in\mathcal{A}(s),
$$

but gradually shifted closer to a deterministic optimal policy.

An $\varepsilon$-greedy policy selects an action with maximal estimated action value with probability $1-\varepsilon$ and selects an action at random with probability $\varepsilon$. For $|\mathcal{A}(s)|$ actions,

$$
\pi(a\mid s)=
\begin{cases}
1-\varepsilon+\dfrac{\varepsilon}{|\mathcal{A}(s)|},
& a=A^*,\\[6pt]
\dfrac{\varepsilon}{|\mathcal{A}(s)|},
& a\ne A^*,
\end{cases}
$$

where

$$
A^*=\arg\max_a Q(s,a).
$$

The $\varepsilon$-greedy policies are examples of $\varepsilon$-soft policies, for which

$$
\pi(a\mid s)\ge\frac{\varepsilon}{|\mathcal{A}(s)|}
$$

for all states and actions. Among $\varepsilon$-soft policies, $\varepsilon$-greedy policies are closest to greedy.

### On-Policy First-Visit MC Control

The policy is improved over the current value function by moving it toward an $\varepsilon$-greedy policy. For any $\varepsilon$-soft policy $\pi$, any $\varepsilon$-greedy policy with respect to $q_\pi$ is guaranteed to be better than or equal to $\pi$.

```text
Algorithm parameter: small epsilon > 0

Initialize:
    pi as an arbitrary epsilon-soft policy
    Q(s, a) arbitrarily, for all s in S and a in A(s)
    Returns(s, a) as an empty list, for all s in S and a in A(s)

Repeat forever, for each episode:
    Generate an episode following pi:
        S_0, A_0, R_1, ..., S_(T-1), A_(T-1), R_T
    G = 0

    For t = T - 1, T - 2, ..., 0:
        G = gamma * G + R_(t+1)

        Unless the pair S_t, A_t appears in
        S_0, A_0, S_1, A_1, ..., S_(t-1), A_(t-1):
            Append G to Returns(S_t, A_t)
            Q(S_t, A_t) = average(Returns(S_t, A_t))
            A* = argmax_a Q(S_t, a)

            For all a in A(S_t):
                pi(a | S_t) = 1 - epsilon + epsilon / |A(S_t)|, if a = A*
                pi(a | S_t) = epsilon / |A(S_t)|, otherwise
```

### Policy Improvement

Let $\pi'$ be the $\varepsilon$-greedy policy. For any $s\in\mathcal{S}$,

$$
\begin{aligned}
q_\pi\bigl(s,\pi'(s)\bigr)
&=\sum_a\pi'(a\mid s)q_\pi(s,a)\\
&=\frac{\varepsilon}{|\mathcal{A}(s)|}\sum_a q_\pi(s,a)
+(1-\varepsilon)\max_a q_\pi(s,a)\\
&\ge
\frac{\varepsilon}{|\mathcal{A}(s)|}\sum_a q_\pi(s,a)
+(1-\varepsilon)
\sum_a
\frac{\pi(a\mid s)-\frac{\varepsilon}{|\mathcal{A}(s)|}}
{1-\varepsilon}
q_\pi(s,a)\\
&=\sum_a\pi(a\mid s)q_\pi(s,a)\\
&=v_\pi(s).
\end{aligned}
$$

The sum replacing the maximum is a weighted average with nonnegative weights summing to $1$, and is no larger than the largest number averaged. Thus, by the policy improvement theorem,

$$
v_{\pi'}(s)\ge v_\pi(s)
\qquad
\text{for all }s\in\mathcal{S}.
$$

Equality can hold only when both policies are optimal among the $\varepsilon$-soft policies. Thus, on-policy Monte Carlo control works for $\varepsilon$-soft policies: every move is toward an $\varepsilon$-greedy policy, except when the best policy among the $\varepsilon$-soft policies has been found.

## Off-Policy Prediction via Importance Sampling

Off-policy methods use two policies:

- The **target policy** $\pi$ is the policy being learned about.
- The **behavior policy** $b$ is the policy used to generate behavior.

The data are generated by $b$, but values are estimated for $\pi$. The target policy may be deterministic, while the behavior policy remains stochastic and more exploratory.

### Assumption of Coverage

To use episodes from $b$ to estimate values for $\pi$, every action taken under $\pi$ must also be taken, at least occasionally, under $b$:

$$
\pi(a\mid s)>0
\quad\Longrightarrow\quad
b(a\mid s)>0.
$$

This is called the assumption of coverage. It follows that $b$ must be stochastic in states where it is not identical to $\pi$.

### Importance-Sampling Ratio

Consider the trajectory beginning at $S_t$ and occurring under policy $\pi$. Its probability is

$$
\Pr\{A_t,S_{t+1},A_{t+1},\ldots,S_T\mid S_t,A_{t:T-1}\sim\pi\}
$$

$$
=\prod_{k=t}^{T-1}
\pi(A_k\mid S_k)
p(S_{k+1}\mid S_k,A_k).
$$

The relative probability of the trajectory under the target and behavior policies is the importance-sampling ratio

$$
\begin{aligned}
\rho_{t:T-1}
&\doteq
\frac{
\prod_{k=t}^{T-1}
\pi(A_k\mid S_k)p(S_{k+1}\mid S_k,A_k)
}{
\prod_{k=t}^{T-1}
b(A_k\mid S_k)p(S_{k+1}\mid S_k,A_k)
}\\
&=
\prod_{k=t}^{T-1}
\frac{\pi(A_k\mid S_k)}{b(A_k\mid S_k)}.
\end{aligned}
$$

The trajectory probabilities depend on the MDP's transition probabilities, but these appear identically in the numerator and denominator and cancel. The ratio depends only on the two policies and the sequence, not on the MDP.

Returns from the behavior policy have the wrong distribution for estimating values under the target policy. Multiplying by the importance-sampling ratio transforms them to have the right expected value:

$$
\mathbb{E}_b\left[\rho_{t:T-1}G_t\mid S_t=s\right]
=v_\pi(s).
$$

### Ordinary Importance Sampling

Let $\mathcal{J}(s)$ be the set of all time steps in which state $s$ is visited, and let $T(t)$ be the first time of termination following time $t$. For first-visit methods, $\mathcal{J}(s)$ includes only time steps that are first visits to $s$ within their episodes.

Ordinary importance sampling uses a simple average of the scaled returns:

$$
V(s)
\doteq
\frac{
\sum_{t\in\mathcal{J}(s)}
\rho_{t:T(t)-1}G_t
}{|\mathcal{J}(s)|}.
$$

The first-visit ordinary importance-sampling estimator is unbiased, but its variance is in general unbounded because the variance of the ratios can be unbounded.

### Weighted Importance Sampling

Weighted importance sampling uses a weighted average:

$$
V(s)
\doteq
\frac{
\sum_{t\in\mathcal{J}(s)}
\rho_{t:T(t)-1}G_t
}{
\sum_{t\in\mathcal{J}(s)}
\rho_{t:T(t)-1}
},
$$

or zero if the denominator is zero.

The first-visit weighted estimator is biased, but the largest weight on any single return is $1$. Assuming bounded returns, its variance converges to zero even if the variance of the ratios is infinite.

Ordinary importance sampling is unbiased and weighted importance sampling is biased, but the weighted estimator typically has dramatically lower variance and is strongly preferred. For every-visit methods, both estimators are biased, although the bias falls asymptotically to zero as the number of samples increases.

## Incremental Implementation

Monte Carlo prediction methods can be implemented incrementally on an episode-by-episode basis. On-policy methods average returns, ordinary importance sampling averages scaled returns, and weighted importance sampling forms a weighted average of returns.

Suppose that $G_1,G_2,\ldots,G_{n-1}$ are returns with corresponding random weights $W_i$, where

$$
W_i=\rho_{t_i:T(t_i)-1}.
$$

The weighted estimate is

$$
V_n
\doteq
\frac{\sum_{k=1}^{n-1}W_kG_k}
{\sum_{k=1}^{n-1}W_k},
\qquad n\ge2.
$$

Maintain the cumulative sum of weights

$$
C_n=\sum_{k=1}^{n}W_k.
$$

After receiving the additional return $G_n$ with weight $W_n$, update

$$
V_{n+1}
\doteq
V_n+\frac{W_n}{C_n}\left[G_n-V_n\right],
\qquad n\ge1,
$$

and

$$
C_{n+1}\doteq C_n+W_{n+1},
$$

where $C_0=0$. The initial value $V_1$ is arbitrary and need not be specified.

### Off-Policy MC Prediction

The following algorithm uses weighted importance sampling to estimate $Q\approx q_\pi$ from episodes generated by a behavior policy $b$ with coverage of $\pi$.

```text
Input: an arbitrary target policy pi

Initialize, for all s in S and a in A(s):
    Q(s, a) arbitrarily
    C(s, a) = 0

Loop forever, for each episode:
    b = any policy with coverage of pi
    Generate an episode following b:
        S_0, A_0, R_1, ..., S_(T-1), A_(T-1), R_T
    G = 0
    W = 1

    For t = T - 1, T - 2, ..., 0, while W != 0:
        G = gamma * G + R_(t+1)
        C(S_t, A_t) = C(S_t, A_t) + W
        Q(S_t, A_t) = Q(S_t, A_t)
                        + W / C(S_t, A_t) * [G - Q(S_t, A_t)]
        W = W * pi(A_t | S_t) / b(A_t | S_t)
```

This algorithm is nominally for off-policy learning using weighted importance sampling, but applies to the on-policy case by choosing the target and behavior policies as the same. Then $b=\pi$ and $W$ is always $1$.

The approximation $Q$ converges to $q_\pi$ for all encountered state-action pairs while actions are selected according to the potentially different policy $b$.

## Off-Policy Monte Carlo Control

In off-policy methods, the policy used to generate behavior, called the **behavior policy** $b$, is separated from the policy that is evaluated and improved, called the **target policy** $\pi$. The target policy may be deterministic, while the behavior policy continues to sample all possible actions.

The behavior policy must have a nonzero probability of selecting all actions that might be selected by the target policy. To explore all possibilities, the behavior policy is soft: it selects all actions in all states with nonzero probability.

The method uses GPI and weighted importance sampling. The target policy $\pi\approx\pi_*$ is the greedy policy with respect to $Q$, which estimates $q_*$:

$$
\pi(s)\doteq\arg\max_a Q(s,a).
$$

The behavior policy $b$ can be any soft policy. To assure convergence of $\pi$ to the optimal policy, an infinite number of returns must be obtained for each state-action pair. This can be assured by choosing $b$ to be $\varepsilon$-soft.

### Off-Policy MC Control Algorithm

```text
Initialize, for all s in S and a in A(s):
    Q(s, a) arbitrarily
    C(s, a) = 0
    pi(s) = argmax_a Q(s, a), with ties broken consistently

Loop forever, for each episode:
    b = any soft policy
    Generate an episode using b:
        S_0, A_0, R_1, ..., S_(T-1), A_(T-1), R_T
    G = 0
    W = 1

    For t = T - 1, T - 2, ..., 0:
        G = gamma * G + R_(t+1)
        C(S_t, A_t) = C(S_t, A_t) + W
        Q(S_t, A_t) = Q(S_t, A_t)
                        + W / C(S_t, A_t) * [G - Q(S_t, A_t)]
        pi(S_t) = argmax_a Q(S_t, a), with ties broken consistently

        If A_t != pi(S_t):
            Exit the inner loop and proceed to the next episode

        W = W / b(A_t | S_t)
```

The policy $\pi$ converges to optimal at all encountered states even though actions are selected according to a different soft policy $b$, which may change between or even within episodes.

A potential problem is that the method learns only from the tails of episodes for which all remaining actions are greedy. If nongreedy actions are frequent, learning will be slow, particularly for states appearing in the early portions of long episodes.

## Discounting-Aware Importance Sampling

The off-policy methods considered so far form importance-sampling weights for returns considered as unitary wholes, without taking into account the returns' internal structures as sums of discounted rewards. This structure can be used to reduce the variance of off-policy estimators.

If $\gamma<1$, factors in the importance-sampling ratio for actions occurring after a reward has already been determined do not change that reward's expected value, but they add to its variance.

### Discounting as Partial Termination

Discounting can be viewed as determining a probability of termination, or a degree of partial termination. For any $\gamma\in[0,1]$, the return $G_0$ can be considered as partly terminating after each step.

Define the flat partial return

$$
\bar{G}_{t:h}
\doteq
R_{t+1}+R_{t+2}+\cdots+R_h,
\qquad 0\le t<h\le T.
$$

The horizon $h$ is the point at which the partial return terminates. The conventional full return can be written as a sum of flat partial returns:

$$
\begin{aligned}
G_t
&=R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}
+\cdots+\gamma^{T-t-1}R_T\\
&=(1-\gamma)R_{t+1}\\
&\quad +(1-\gamma)\gamma(R_{t+1}+R_{t+2})\\
&\quad +(1-\gamma)\gamma^2(R_{t+1}+R_{t+2}+R_{t+3})\\
&\quad +\cdots\\
&\quad +(1-\gamma)\gamma^{T-t-2}
(R_{t+1}+R_{t+2}+\cdots+R_{T-1})\\
&\quad +\gamma^{T-t-1}
(R_{t+1}+R_{t+2}+\cdots+R_T)\\
&=(1-\gamma)
\sum_{h=t+1}^{T-1}
\gamma^{h-t-1}\bar{G}_{t:h}
+\gamma^{T-t-1}\bar{G}_{t:T}.
\end{aligned}
$$

A flat partial return $\bar{G}_{t:h}$ includes rewards only up to horizon $h$. It therefore requires only the importance-sampling ratio up to $h-1$.

### Ordinary Discounting-Aware Estimator

$$
V(s)
\doteq
\frac{1}{|\mathcal{J}(s)|}
\sum_{t\in\mathcal{J}(s)}
\left(
(1-\gamma)
\sum_{h=t+1}^{T(t)-1}
\gamma^{h-t-1}
\rho_{t:h-1}\bar{G}_{t:h}
+
\gamma^{T(t)-t-1}
\rho_{t:T(t)-1}\bar{G}_{t:T(t)}
\right).
$$

### Weighted Discounting-Aware Estimator

$$
V(s)
\doteq
\frac{
\sum_{t\in\mathcal{J}(s)}
\left(
(1-\gamma)
\sum_{h=t+1}^{T(t)-1}
\gamma^{h-t-1}
\rho_{t:h-1}\bar{G}_{t:h}
+
\gamma^{T(t)-t-1}
\rho_{t:T(t)-1}\bar{G}_{t:T(t)}
\right)
}{
\sum_{t\in\mathcal{J}(s)}
\left(
(1-\gamma)
\sum_{h=t+1}^{T(t)-1}
\gamma^{h-t-1}
\rho_{t:h-1}
+
\gamma^{T(t)-t-1}
\rho_{t:T(t)-1}
\right)
}.
$$

These discounting-aware importance-sampling estimators take discount into account. If $\gamma=1$, they are the same as the off-policy estimators from the previous section.

## Per-Decision Importance Sampling

The structure of the return as a sum of rewards can be taken into account in off-policy importance sampling to reduce variance, even when $\gamma=1$.

In the ordinary off-policy estimators, each term of the numerator is itself a sum:

$$
\begin{aligned}
\rho_{t:T-1}G_t
&=\rho_{t:T-1}
\left(
R_{t+1}+\gamma R_{t+2}+\cdots+
\gamma^{T-t-1}R_T
\right)\\
&=\rho_{t:T-1}R_{t+1}
+\gamma\rho_{t:T-1}R_{t+2}
+\cdots+
\gamma^{T-t-1}\rho_{t:T-1}R_T.
\end{aligned}
$$

For the first sub-term,

$$
\rho_{t:T-1}R_{t+1}
=
\frac{\pi(A_t\mid S_t)}{b(A_t\mid S_t)}
\frac{\pi(A_{t+1}\mid S_{t+1})}{b(A_{t+1}\mid S_{t+1})}
\cdots
\frac{\pi(A_{T-1}\mid S_{T-1})}{b(A_{T-1}\mid S_{T-1})}
R_{t+1}.
$$

Only the first factor and the reward are related; the other factors are for events that occur after the reward. The expected value of each later factor is one:

$$
\mathbb{E}
\left[
\frac{\pi(A_k\mid S_k)}{b(A_k\mid S_k)}
\right]
=
\sum_a b(a\mid S_k)
\frac{\pi(a\mid S_k)}{b(a\mid S_k)}
=
\sum_a\pi(a\mid S_k)
=1.
$$

Thus, the later factors have no effect in expectation:

$$
\mathbb{E}
\left[
\rho_{t:T-1}R_{t+1}
\right]
=
\mathbb{E}
\left[
\rho_{t:t}R_{t+1}
\right].
$$

For the $k$th sub-term,

$$
\mathbb{E}
\left[
\rho_{t:T-1}R_{t+k}
\right]
=
\mathbb{E}
\left[
\rho_{t:t+k-1}R_{t+k}
\right].
$$

It follows that

$$
\mathbb{E}
\left[
\rho_{t:T-1}G_t
\right]
=
\mathbb{E}
\left[
\tilde{G}_t
\right],
$$

where

$$
\tilde{G}_t
=\rho_{t:t}R_{t+1}
+\gamma\rho_{t:t+1}R_{t+2}
+\gamma^2\rho_{t:t+2}R_{t+3}
+\cdots+
\gamma^{T-t-1}\rho_{t:T-1}R_T.
$$

This is called **per-decision importance sampling**.

The corresponding ordinary importance-sampling estimator is

$$
V(s)
\doteq
\frac{
\sum_{t\in\mathcal{J}(s)}\tilde{G}_t
}{|\mathcal{J}(s)|},
$$

which may sometimes have lower variance. A per-decision version of weighted importance sampling is less clear; the estimators proposed so far are not consistent.